# 🔒 Notebook 1: A Lock With No Expiry (the BAD way)

**The problem:** in a distributed system one node holds a lock to do exclusive work — flush a cache, run a cron job, become the leader. Now the holder **crashes** before releasing it.

If the lock has no expiry, *no one else can ever get it*.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/lease
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: a plain in-memory lock

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class SimpleLock:
    holder: Optional[str] = None

    def acquire(self, who: str) -> bool:
        if self.holder is None:
            self.holder = who
            return True
        return False

    def release(self, who: str):
        if self.holder == who:
            self.holder = None

lock = SimpleLock()
print('node-A acquires:', lock.acquire('node-A'))
# node-A crashes here without calling release()
print('node-B tries:    ', lock.acquire('node-B'))
print('holder is still:', lock.holder)


Node B is locked out **forever**, even though node A is dead. We need a way for the lock to *expire on its own*.

👉 Next notebook: **leases** — locks with a time-to-live the holder must renew.